In [1]:
!pip install transformers sentencepiece scikit-learn -q

In [2]:
import os
import torch
import warnings
import logging
import numpy as np
import pandas as pd
from transformers import (
    pipeline,
    AutoTokenizer, AutoModelForSeq2SeqLM,
    BertTokenizer, BertForSequenceClassification
)
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from google.colab import drive
import time
from tqdm import tqdm

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    print(f"Using GPU: {torch.cuda.get_device_name(device)}")

Using GPU: NVIDIA A100-SXM4-40GB


In [4]:
warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger().setLevel(logging.ERROR)

from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

drive.mount('/content/drive', force_remount=True)

drive_path = '/content/drive/MyDrive/Event Log/2025-7-28 申毅实习/策略研究/NLP 日度'
os.chdir(drive_path)

Mounted at /content/drive


In [5]:
def add_exchange_suffix(code: str) -> str:
    code = str(code)

    if len(code) == 5 and code.isdigit():
        ticker = f"{code}.HK"
        return ticker

    if code.startswith(("60", "68", "900")):
        ticker = f"{code}.SH"
        return ticker

    if code.startswith(("00", "30", "200")):
        ticker = f"{code}.SZ"
        return ticker

    return code

In [6]:
device = 0 if torch.cuda.is_available() else -1
print("Device set to", "GPU" if device == 0 else "CPU")

Device set to GPU


In [7]:
nllb_model_name = "facebook/nllb-200-3.3B"
nllb_tokenizer = AutoTokenizer.from_pretrained(nllb_model_name)
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(nllb_model_name)

translator = pipeline(
    "translation",
    model=nllb_model,
    tokenizer = nllb_tokenizer,
    src_lang = "zho_Hans",
    tgt_lang = "eng_Latn",
    device=device,
    truncation = True,
    max_length = 512
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

pytorch_model-00001-of-00003.bin:   0%|          | 0.00/6.93G [00:00<?, ?B/s]

pytorch_model-00002-of-00003.bin:   0%|          | 0.00/8.55G [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [8]:
finbert_model = "yiyanghkust/finbert-tone"
tokenizer = BertTokenizer.from_pretrained(finbert_model)
model = BertForSequenceClassification.from_pretrained(finbert_model)

classifier = pipeline(
    "sentiment-analysis",
    model = model,
    tokenizer = tokenizer,
    device = device
)

vocab.txt: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/533 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

In [7]:
equity_reports = pd.read_csv(
    'equity_reports_1.csv',
    encoding='utf-8',
    parse_dates=['PUBLISH_DATE', 'UPDATE_TIME', 'UPDATE_DATE'],
    dtype={"STOCK_CODE": str}
)

equity_reports['PUBLISH_DATE'] = pd.to_datetime(equity_reports['PUBLISH_DATE'], errors='coerce')
equity_reports['UPDATE_TIME']  = pd.to_datetime(equity_reports['UPDATE_TIME'], errors='coerce')
equity_reports['UPDATE_DATE']  = pd.to_datetime(equity_reports['UPDATE_DATE'], errors='coerce')

equity_reports['STOCK_CODE'] = equity_reports['STOCK_CODE'].apply(add_exchange_suffix)

print(len(equity_reports))
equity_reports.head()

10908


,REPORT_ID,SUMMARY,TITLE,PUBLISH_DATE,STOCK_CODE,STOCK_ABBR,UPDATE_TIME,UPDATE_DATE,Lag
0,5027056,事件：公司公布2022年三季报，前三季度实现收入43.84亿元（+12.91%）；实现归母净...,洽洽食品（002557）2022年三季报点评：产品提价红利显现，盈利能力逐步修复,2022-10-27,002557.SZ,洽洽食品,2022-10-28 10:02:38,2022-10-28,1
1,5027057,事件：宁波银行披露2022年三季报，今年前三季度营收增速为15.21%，拨备前利润增速为15...,宁波银行（002142）2022年三季报点评：净息差改善，资产质量优异,2022-10-27,002142.SZ,宁波银行,2022-10-28 10:02:05,2022-10-28,1
2,5027068,主要观点：事件：公司发布2022年第三季度报告2022Q1-Q3公司实现营业收入43.84亿...,洽洽食品（002557）：Q3毛利修复费用增投，盈利能力仍有所承压,2022-10-27,002557.SZ,洽洽食品,2022-10-28 10:00:30,2022-10-28,1
3,5027073,事件：公司发布2022年三季报，22年前三季度营收557.8亿，同比增长12.19%，归母净...,五粮液（000858）22Q3点评：业绩超预期，增长仍稳健,2022-10-27,000858.SZ,五粮液,2022-10-28 10:00:45,2022-10-28,1
4,5027074,疫情波动下业绩承压，关注业态升级&amp;线上数字化打开成长新动能事件：公司发布2022三季...,天虹股份（002419）：疫情波动下业绩承压，关注业态升级、线上数字化打开成长新动能,2022-10-27,002419.SZ,天虹股份,2022-10-28 14:55:16,2022-10-28,1


In [10]:
sentiment_map = {
    'POSITIVE': '正面',
    'NEUTRAL': '中性',
    'NEGATIVE': '负面'
}

output_path = "sentiment_output.csv"
buffer = []
batch_size = 100
batch_bar = None

if os.path.exists(output_path):
    df_existing = pd.read_csv(output_path, usecols=['REPORT_ID'])
    processed_ids = set(df_existing['REPORT_ID'])
    print(f"Progress Found: {len(processed_ids)} Finished")
else:
    pd.DataFrame(columns=[
        'REPORT_ID', 'title_label', 'title_score', 'summary_label', 'summary_score'
    ]).to_csv(output_path, index=False)
    processed_ids = set()
    print("Progress Log Created")

Progress Found: 88180 Finished


In [11]:
date_ranges = [
    (pd.Timestamp("2022-01-01"), pd.Timestamp("2022-06-01"))
]

def is_within_date_ranges(date, date_ranges):
    if pd.isna(date):
        return False
    for start, end in date_ranges:
        if start <= date < end:
            return True
    return False

In [ ]:
for i, row in equity_reports.iterrows():
    ID = row['REPORT_ID']
    if ID in processed_ids:
        continue

    # if not is_within_date_ranges(row['UPDATE_TIME'], date_ranges):
    #     continue

    ticker = str(row['STOCK_CODE'])

    if not (ticker.endswith('.SH') or ticker.endswith('.SZ')):
        continue

    if batch_bar is None:
        batch_bar = tqdm(total=batch_size, desc=f"Current Batch", leave=False)

    title_cn = row['TITLE']
    summary_cn = row['SUMMARY']

    try:
        if pd.notna(title_cn):
            title_en = translator(title_cn, max_length=600, truncation=True)[0]['translation_text']
            title_output = classifier(title_en, truncation=True, max_length=512)
            title_label = title_output[0]['label'] if title_output else None
            title_score = title_output[0]['score'] if title_output else None
        else:
            title_label = None
            title_score = None

        if pd.notna(summary_cn):
            summary_en = translator(summary_cn, max_length=1024, truncation=True)[0]['translation_text']
            summary_output = classifier(summary_en, truncation=True, max_length=512)
            summary_label = summary_output[0]['label'] if summary_output else None
            summary_score = summary_output[0]['score'] if summary_output else None
        else:
            summary_label = None
            summary_score = None

        buffer.append({
            'REPORT_ID': ID,
            'title_label': title_label,
            'title_score': title_score,
            'summary_label': summary_label,
            'summary_score': summary_score
        })

        batch_bar.update(1)

        if len(buffer) >= batch_size:
            df_batch = pd.DataFrame(buffer)
            df_batch.to_csv(output_path, mode='a', header=False, index=False)
            print(f"[Row {i}] → Dumped {batch_size} rows to {output_path}")
            buffer.clear()
            batch_bar.close()
            batch_bar = None

    except Exception as e:
        print(f"Error at index {i} (REPORT_ID={ID}): {e}")

if buffer:
    df_batch = pd.DataFrame(buffer)
    df_batch.to_csv(output_path, mode='a', header=False, index=False)
    buffer.clear()

    if batch_bar is None:
        batch_bar = tqdm(total=batch_size, desc=f"Current Batch", leave=False)

[Row 99] → Dumped 100 rows to sentiment_output.csv


[Row 199] → Dumped 100 rows to sentiment_output.csv


Current Batch:  81%|████████  | 81/100 [16:55<02:46,  8.76s/it]

In [ ]:
!kill -9 -1

In [14]:
count = 0

for i, row in equity_reports.iterrows():
    ID = row['REPORT_ID']
    if ID in processed_ids:
        continue

    if not is_within_date_ranges(row['UPDATE_TIME'], date_ranges):
        continue

    ticker = str(row['STOCK_CODE'])

    if not (ticker.endswith('.SH') or ticker.endswith('.SZ')):
        continue

    count += 1

count

0

In [ ]:
equity_reports.columns

Index(['REPORT_ID', 'SUMMARY', 'TITLE', 'PUBLISH_DATE', 'STOCK_CODE',
       'STOCK_ABBR', 'UPDATE_TIME', 'UPDATE_DATE', 'Lag'],
      dtype='object')

In [ ]:
valid_count = 0

for i, row in equity_reports.iterrows():
    if is_within_date_ranges(row['UPDATE_TIME'], date_ranges) and row['REPORT_ID'] not in processed_ids:
        valid_count += 1

print(f"Total valid rows: {valid_count}")

Total valid rows: 5737


In [ ]:
equity_reports.head()

,REPORT_ID,SUMMARY,TITLE,PUBLISH_DATE,STOCK_CODE,STOCK_ABBR,UPDATE_TIME,UPDATE_DATE,Lag
0,4408114,投资要点事件：近日，（1）国家药监局批准公司的1类创新药奥布替尼（商品名：宜诺凯）上...,诺诚健华-B（09969）：奥布替尼两项淋巴瘤适应症获批，纳入港股通期待估值修复,2020-12-29,09969.HK,诺诚健华-B,2021-05-19 17:39:59,2021-05-19,141
1,4408146,战略明确，长期成长路径清晰，维持“买入”评级无论是中端期，还是长期视角看，公司均表现出优异的...,金山办公（688111）公司深度报告之二：循成长之迹，探WPS发展远景,2020-12-29,688111.SH,金山办公,2020-12-29 09:32:28,2020-12-29,0
2,4408165,股权激励划定明确增长目标，有望充分激发公司市值与成长动力。公司发布2021年限制性股票激励计...,共创草坪（605099）：股权激励划定明确目标，人造草坪龙头再添增长动力,2020-12-29,605099.SH,共创草坪,2021-03-09 12:15:20,2021-03-09,70
3,4408166,事件：公司拟收购合肥龙翔高复学校70%股权。公司于2020年12月28日与毛成红、合肥龙翔高...,科德教育（300192）：拟并购合肥龙翔高复学校，坚定推进教育产业布局,2020-12-29,300192.SZ,科德教育,2021-03-09 12:15:21,2021-03-09,70
4,4408220,NaN,华住集团-S（01179）跟踪报告：流量品牌技术三位一体，引领行业发展,2020-12-29,01179.HK,华住集团-S,2021-02-03 16:50:49,2021-02-03,36


In [18]:
print(title_cn)

华锐精密（688059）2022年中报点评：产能爬坡在即，助力业绩兑现与市占率提升


In [19]:
print(title_en)

Huawei Precision ((688059)) Mid-2022 report comments: Capacity climb is imminent, assist performance is achieved and market share is improved
